In [3]:
import numpy as np
import os
import tensorflow as tf
from matplotlib import pyplot as plt
import mediapipe as mp
import cv2
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (LSTM, Dense, Dropout, BatchNormalization,
                                     Bidirectional, Masking, LayerNormalization)
from tensorflow.keras.callbacks import TensorBoard, EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import Adam
from tensorflow.keras import regularizers
import tensorflow.keras.backend as K

In [5]:
# ─── Cấu hình dataset ─────────────────────────────────────────────────────────
DATA_PATH = os.path.join('DATA-FINAL')

## 60 video dữ liệu
num_sequences = 100
#mỗi video dài 30 frame
sequence_length = 30

In [7]:
actions = np.array(['An','Ban','Bien','Binh Minh','Buoi Sang','Buoi Toi','Buon','Ca Phe','Cai Gi','Cam On','Cam Thay','Cua','Dau Dau','Dep','Di',
                   'Gia Lai','Hoang Hon','Hoc','Hom Nay','Hom Qua','Khoe','Khong','Khong Muon','Khong Thich','Lo Lang','Mat','Met','Mua','Muon','Nang','Ngam','Ngay Mai',
                   'Nghe','Nhac','Nhat Ban','Nhung','Nong','Quy Nhon','Song','Ten','Thich','Toi','Toi Thich Ban','Tom','Troi','Uong','Viet Nam','Vui','Xin Chao','Xin Loi'])

In [9]:
label_map = {label: num for num, label in enumerate(actions)}

In [11]:
label_map

{'An': 0,
 'Ban': 1,
 'Bien': 2,
 'Binh Minh': 3,
 'Buoi Sang': 4,
 'Buoi Toi': 5,
 'Buon': 6,
 'Ca Phe': 7,
 'Cai Gi': 8,
 'Cam On': 9,
 'Cam Thay': 10,
 'Cua': 11,
 'Dau Dau': 12,
 'Dep': 13,
 'Di': 14,
 'Gia Lai': 15,
 'Hoang Hon': 16,
 'Hoc': 17,
 'Hom Nay': 18,
 'Hom Qua': 19,
 'Khoe': 20,
 'Khong': 21,
 'Khong Muon': 22,
 'Khong Thich': 23,
 'Lo Lang': 24,
 'Mat': 25,
 'Met': 26,
 'Mua': 27,
 'Muon': 28,
 'Nang': 29,
 'Ngam': 30,
 'Ngay Mai': 31,
 'Nghe': 32,
 'Nhac': 33,
 'Nhat Ban': 34,
 'Nhung': 35,
 'Nong': 36,
 'Quy Nhon': 37,
 'Song': 38,
 'Ten': 39,
 'Thich': 40,
 'Toi': 41,
 'Toi Thich Ban': 42,
 'Tom': 43,
 'Troi': 44,
 'Uong': 45,
 'Viet Nam': 46,
 'Vui': 47,
 'Xin Chao': 48,
 'Xin Loi': 49}

In [13]:
sequences, labels= [], [] #Danh sách chứa toàn bộ dữ liệu video vvà nhãn
for action in actions:
    for sequence in range(num_sequences):
        window = []  #Lưu 30 frame của 1 video
        for frame_num in range(sequence_length):
            res = np.load(os.path.join(DATA_PATH,action, str(sequence), '{}.npy'.format(frame_num)))
            window.append(res)
        sequences.append(window)
        labels.append(label_map[action])

In [15]:
X = np.array(sequences, dtype=np.float32)
y = np.array(labels)

In [17]:
np.save('sequences.npy', X)
np.save('labels.npy', y)

In [19]:
X = np.load('sequences.npy')
y_int = np.load('labels.npy')

In [21]:
y_int = np.load('labels.npy')

In [23]:
y = to_categorical(y_int)

In [25]:
X.shape

(5000, 30, 1662)

In [27]:
# ========================
# 1. Cắt pose + hands
# ========================
pose = X[:, :, 0:132]

lh = X[:, :, 1536:1599]
rh = X[:, :, 1599:1662]

# ========================
# 2. Face indices 
# ========================
#Viền mặt
idx_vien_mat = [
    10, 338, 297, 332, 284, 251, 389, 356,
    454, 323, 361, 288, 397, 365, 379, 378,
    400, 377, 152, 148, 176, 149, 150, 136,
    172, 58, 132, 93, 234, 127, 162, 21,
    54, 103, 67, 109
]

#Mắt
idx_mat_trai = [33,160,158,133,153,144,145,163,7,246,226,110, 24,23,22,26,112,243,190,56,28,27,29,30,247]
idx_mat_phai = [362,385,387,263,373,380,374,390,249,466,463,341,256,252,253,254,339,446,467,260,259,257,258,286,414]

#Lông mày
idx_long_may_trai = [70,63,105,66,107,55,65,52,53,46]
idx_long_may_phai = [336,296,334,293,300,276,283,282,295,285]

idx_mieng = [
    61,146,91,181,84,17,314,405,
    321,375,291,409,270,269,267,0,37,39,40,185,308,324,318,402,317,
    14,87,178,88,95,78,191,80,81,82,13,312,311,310,415
]

SELECTED_FACE_IDX = sorted(set(
    idx_vien_mat +
    idx_mat_trai + idx_mat_phai +
    idx_long_may_trai + idx_long_may_phai +
    idx_mieng
))

# ========================
# 3. Convert sang index trong X
# ========================
# Face bắt đầu từ index 132, mỗi điểm có 3 giá trị (x,y,z)

face_features = []

for idx in SELECTED_FACE_IDX:
    start = 132 + idx * 3
    face_features.extend([start, start+1, start+2])

face_features = np.array(face_features)
# ========================
# 4. Extract face subset
# ========================
face = X[:, :, face_features]

# ========================
# 5. Concatenate lại
# ========================
X_new = np.concatenate([pose, face, lh, rh], axis=2)

print("Shape sau cắt:", X_new.shape)

# ========================
# 6. Save
# ========================
np.save('sequences_selected-t5.npy', X_new)

Shape sau cắt: (5000, 30, 696)


In [ ]:
# X = np.load('sequences_selected-t5.npy')

In [29]:
X.shape

(5000, 30, 1662)

In [31]:
def standardize_keypoints(X_data):
    X_cent = X_data.copy()
    is_2d = False
    if len(X_cent.shape) == 2:
        is_2d = True
        X_cent = np.expand_dims(X_cent, axis=0) 
        
    # --- 1. Nội suy 2 chiều (Tiến và Lùi) để sạch bóng giá trị 0 ---
    for b in range(X_cent.shape[0]):
        # Tiến (Forward fill)
        for t in range(1, X_cent.shape[1]):
            zeros_mask = (X_cent[b, t, :] == 0.0)
            X_cent[b, t, zeros_mask] = X_cent[b, t-1, zeros_mask]
        # Lùi (Backward fill - xử lý nốt nếu frame 0 bị lỗi)
        for t in range(X_cent.shape[1]-2, -1, -1):
            zeros_mask = (X_cent[b, t, :] == 0.0)
            X_cent[b, t, zeros_mask] = X_cent[b, t+1, zeros_mask]
            
    # --- 2. Chống rung EMA (Giữ nguyên) ---
    alpha = 0.6  
    for b in range(X_cent.shape[0]):
        for t in range(1, X_cent.shape[1]):
            X_cent[b, t, :] = alpha * X_cent[b, t, :] + (1 - alpha) * X_cent[b, t-1, :]
            
    # --- 3. Centering an toàn (Neo điểm) ---
    nose_x = X_cent[:, :, 0:1].copy()
    nose_y = X_cent[:, :, 1:2].copy()
    nose_z = X_cent[:, :, 2:3].copy()
    
    # Dự phòng: Nếu mũi Pose bị 0, dùng mũi của Face (index 132)
    mask_zero = (nose_x == 0)
    nose_x[mask_zero] = X_cent[:, :, 132:133][mask_zero]
    nose_y[mask_zero] = X_cent[:, :, 133:134][mask_zero]
    nose_z[mask_zero] = X_cent[:, :, 134:135][mask_zero]
    
    X_cent[:, :, 0:132:4] -= nose_x
    X_cent[:, :, 1:132:4] -= nose_y
    X_cent[:, :, 2:132:4] -= nose_z
    X_cent[:, :, 132::3] -= nose_x
    X_cent[:, :, 133::3] -= nose_y
    X_cent[:, :, 134::3] -= nose_z
    
    # --- 4. Scaling (Giữ nguyên logic của bạn) ---
    # ... (giữ nguyên phần tính dist và chia mean_dist)
    
    return X_cent


X = np.load('sequences_selected-t5.npy', allow_pickle=True)
print("Shape ban đầu:", X.shape)
X = standardize_keypoints(X)
print("Shape sau chuẩn hóa:", X.shape)

Shape ban đầu: (5000, 30, 696)
Shape sau chuẩn hóa: (5000, 30, 696)


In [33]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.15, random_state=42,stratify=y_int)

In [35]:
X_train.shape

(4250, 30, 696)

In [37]:
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train,
    test_size=0.176,  # 0.125 * 0.8 = 0.1 (10% tổng)
    random_state=42,
    stratify=y_train.argmax(axis=1)  
)

In [39]:
X_train.shape

(3502, 30, 696)

In [ ]:
#**Các kĩ thuật tăng cường dữ liệu**

In [41]:
def augment_sequence(seq):
    aug = seq.copy()
    zero_mask = (seq == 0.0) # Lưu lại mặt nạ khoảng trống

    # 1. Thêm nhiễu
    if np.random.rand() < 0.5:
        aug += np.random.normal(0, 0.01, aug.shape) 

    # 2. Dịch chuyển thời gian
    if np.random.rand() < 0.5:
        shift = np.random.randint(-4, 5) 
        if shift > 0: 
            aug[shift:] = aug[:-shift]
            aug[:shift] = 0.0  
        elif shift < 0: 
            aug[:shift] = aug[-shift:]
            aug[shift:] = 0.0 

    # 3. Scaling nhỏ
    if np.random.rand() < 0.5:
        scale = np.random.uniform(0.9, 1.1)
        aug = aug * scale

    # Trả các khoảng trống về lại đúng 0.0
    aug[zero_mask] = 0.0 

    return aug.astype(np.float32)


In [ ]:
**Tạo thêm dữ liệu giả (augment) từ dữ liệu gốc**

In [43]:
def build_augmented_dataset(X, y, n_aug=2):
    X_list = [X.astype(np.float32)]
    y_list = [y]
    for _ in range(n_aug):
        X_list.append(np.array([augment_sequence(x) for x in X], dtype=np.float32))
        y_list.append(y)
    X_aug = np.concatenate(X_list)
    y_aug = np.concatenate(y_list)
    perm  = np.random.permutation(len(X_aug))
    return X_aug[perm], y_aug[perm]

In [45]:
X_train_aug, y_train_aug = build_augmented_dataset(X_train, y_train, n_aug=2)
print(f"Train sau augment: {X_train_aug.shape}")

Train sau augment: (10506, 30, 696)


In [47]:
# Shuffle lại sau augment
perm = np.random.permutation(len(X_train_aug))
X_train_aug, y_train_aug = X_train_aug[perm], y_train_aug[perm]

In [49]:
X_train_aug.shape

(10506, 30, 696)

In [ ]:
#**CHUẨN HÓA INPUT (Normalization)**

In [51]:
# Chuẩn hóa input VỀ mean ~ VÀ std ~ 1 -> Giúp model học ổn định và nhanh hơn
#Giúp model học ổn định và nhanh hơn
# Fit scaler chỉ trên train, transform cả train + test
mean = X_train_aug.mean(axis=(0,1), keepdims=True)
std  = X_train_aug.std(axis=(0,1),  keepdims=True) + 1e-8

X_train_norm = (X_train_aug - mean) / std
X_test_norm  = (X_test  - mean) / std
X_val_norm   = (X_val  - mean) / std

X_train_norm[X_train_aug == 0.0] = 0.0
X_test_norm[X_test == 0.0] = 0.0
X_val_norm[X_val == 0.0] = 0.0

np.save('norm_mean.npy', mean.squeeze())
np.save('norm_std.npy',  std.squeeze())


In [53]:
num_classes = y.shape[1]
model = Sequential([
    LayerNormalization(input_shape=(30, 696)), 
    
    LSTM(128, return_sequences=True, activation='tanh', recurrent_dropout=0.2),
    BatchNormalization(),
    Dropout(0.4),

    LSTM(64, return_sequences=False, activation='tanh', recurrent_dropout=0.2),
    BatchNormalization(),
    Dropout(0.4),

    Dense(128, activation='relu', kernel_regularizer=regularizers.l2(0.005)),
    Dropout(0.3),

    Dense(num_classes, activation='softmax')
])


In [55]:
initial_lr   = 0.001
decay_steps  = 5000
lr_schedule  = tf.keras.optimizers.schedules.CosineDecayRestarts(
    initial_learning_rate=initial_lr,
    first_decay_steps=decay_steps,
    t_mul=2.0, m_mul=0.9, alpha=1e-5
)

optimizer = Adam(learning_rate=lr_schedule)
loss      = tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.05)  # 0.1→0.05

model.compile(optimizer=optimizer, loss=loss, metrics=['categorical_accuracy'])
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 layer_normalization (Layer  (None, 30, 696)           1392      
 Normalization)                                                  
                                                                 
 lstm (LSTM)                 (None, 30, 128)           422400    
                                                                 
 batch_normalization (Batch  (None, 30, 128)           512       
 Normalization)                                                  
                                                                 
 dropout (Dropout)           (None, 30, 128)           0         
                                                                 
 lstm_1 (LSTM)               (None, 64)                49408     
                                                                 
 batch_normalization_1 (Bat  (None, 64)                2

In [57]:
# ─── Callbacks ────────────────────────────────────────────────────────────────
log_dir = os.path.join('Logs')
callbacks = [
    TensorBoard(log_dir=log_dir),
    EarlyStopping(
        monitor='val_loss',
        patience=30,
        restore_best_weights=True,
        mode='min',
        verbose=1
    ),
    #Lưu model tốt nhất
    ModelCheckpoint(
        'best_model.keras',
        monitor='val_loss',
        save_best_only=True,
        mode='min',
        verbose=1
    )
]

In [59]:
model.fit(
    X_train_norm, y_train_aug,
    epochs=1000,
    batch_size=32,                          
    validation_data=(X_val_norm, y_val),   
    callbacks=callbacks,       
)

Epoch 1/1000


329/329 [==============================] - ETA: 0s - loss: 3.3251 - categorical_accuracy: 0.2420
Epoch 1: val_loss improved from inf to 2.23517, saving model to best_model.keras
329/329 [==============================] - 61s 143ms/step - loss: 3.3251 - categorical_accuracy: 0.2420 - val_loss: 2.2352 - val_categorical_accuracy: 0.5521
Epoch 2/1000
329/329 [==============================] - ETA: 0s - loss: 2.2578 - categorical_accuracy: 0.4867
Epoch 2: val_loss improved from 2.23517 to 1.60352, saving model to best_model.keras
329/329 [==============================] - 53s 160ms/step - loss: 2.2578 - categorical_accuracy: 0.4867 - val_loss: 1.6035 - val_categorical_accuracy: 0.7099
Epoch 3/1000
329/329 [==============================] - ETA: 0s - loss: 1.7556 - categorical_accuracy: 0.6459
Epoch 3: val_loss improved from 1.60352 to 1.28976, saving model to best_model.keras
329/329 [==============================] - 48s 146ms/step - loss: 1.7556 - categorical_accuracy: 0.64

In [65]:
import numpy as np
from tensorflow.keras.models import load_model

model = load_model("best_model.keras")

val_loss, val_acc = model.evaluate(X_test_norm, y_test, verbose=0)
print(f"Độ chính xác: {val_acc*100:.1f}%")

Độ chính xác: 96.8%


In [63]:
del model

In [67]:
from tensorflow.keras.models import load_model

model = load_model("best_model.keras")

# Đánh giá trên tập kiểm tra
test_loss, test_acc = model.evaluate(X_test_norm, y_test, verbose=0)
print(f"Loss:     {test_loss:.4f}")
print(f"Accuracy: {test_acc*100:.2f}%")


Loss:     0.5882
Accuracy: 96.80%


In [73]:
from sklearn.metrics import precision_score, recall_score, f1_score

y_pred = model.predict(X_test_norm)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true_classes = np.argmax(y_test, axis=1)

precision = precision_score(y_true_classes, y_pred_classes, average='weighted')
recall    = recall_score(y_true_classes, y_pred_classes, average='weighted')
f1        = f1_score(y_true_classes, y_pred_classes, average='weighted')

print(f"Precision: {precision*100:.2f}%")
print(f"Recall:    {recall*100:.2f}%")
print(f"F1-Score:  {f1*100:.2f}%")


24/24 [==============================] - 1s 39ms/step
Precision: 96.96%
Recall:    96.80%
F1-Score:  96.74%


In [69]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

# Dự đoán
y_pred = model.predict(X_test_norm)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true_classes = np.argmax(y_test, axis=1)

# In báo cáo chi tiết từng lớp
print(classification_report(y_true_classes, y_pred_classes, target_names=actions))


24/24 [==============================] - 2s 32ms/step
               precision    recall  f1-score   support

           An       1.00      1.00      1.00        15
          Ban       0.94      1.00      0.97        15
         Bien       1.00      1.00      1.00        15
    Binh Minh       1.00      1.00      1.00        15
    Buoi Sang       1.00      1.00      1.00        15
     Buoi Toi       1.00      1.00      1.00        15
         Buon       0.94      1.00      0.97        15
       Ca Phe       1.00      0.93      0.97        15
       Cai Gi       1.00      1.00      1.00        15
       Cam On       1.00      1.00      1.00        15
     Cam Thay       0.93      0.87      0.90        15
          Cua       0.88      0.93      0.90        15
      Dau Dau       1.00      1.00      1.00        15
          Dep       0.94      1.00      0.97        15
           Di       1.00      1.00      1.00        15
      Gia Lai       1.00      1.00      1.00        15
    Hoang 

In [71]:
from sklearn.metrics import precision_score, recall_score, f1_score

y_pred = model.predict(X_test_norm)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true_classes = np.argmax(y_test, axis=1)

precision = precision_score(y_true_classes, y_pred_classes, average='weighted')
recall    = recall_score(y_true_classes, y_pred_classes, average='weighted')
f1        = f1_score(y_true_classes, y_pred_classes, average='weighted')

print(f"Precision: {precision*100:.2f}%")
print(f"Recall:    {recall*100:.2f}%")
print(f"F1-Score:  {f1*100:.2f}%")


24/24 [==============================] - 1s 36ms/step
Precision: 96.96%
Recall:    96.80%
F1-Score:  96.74%
